In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/ReviewsMidExcludedEqualSamplesSplit.zip -d /content/data

In [ ]:
import numpy as np

import tensorflow as tf
from tensorflow.keras import utils

In [ ]:
BATCH_SIZE = 32
SEED = 42
# VOCAB_SIZE = 5000

In [ ]:
train_data_directory = '/content/data/ReviewsMidExcludedEqualSamplesSplit/train'

In [ ]:
raw_training_dataset = utils.text_dataset_from_directory(
    train_data_directory,
    batch_size=BATCH_SIZE,
    seed=SEED,
    validation_split=0.2,
    subset='training')

Found 12366 files belonging to 2 classes.
Using 9893 files for training.


In [ ]:
for i, label in enumerate(raw_training_dataset.class_names):
  print("Label", i, "corresponds to", label)

Label 0 corresponds to high_80
Label 1 corresponds to low_50


In [ ]:
raw_validation_dataset = utils.text_dataset_from_directory(
    train_data_directory,
    batch_size=BATCH_SIZE,
    seed=SEED,
    validation_split=0.2,
    subset='validation'
)

Found 12366 files belonging to 2 classes.
Using 2473 files for validation.


In [ ]:
test_data_directory = '/content/data/ReviewsMidExcludedEqualSamplesSplit/test'
raw_testing_dataset = utils.text_dataset_from_directory(
    test_data_directory,
    batch_size=BATCH_SIZE
)

Found 3092 files belonging to 2 classes.


In [ ]:
raw_training_dataset_optimized = raw_training_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
raw_validation_dataset_optimized = raw_validation_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
raw_testing_dataset_optimized = raw_testing_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.layers import TextVectorization

In [ ]:
# encoder = TextVectorization(max_tokens=VOCAB_SIZE)
encoder = TextVectorization()
encoder.adapt(raw_training_dataset_optimized.map(lambda text, label: text))

In [ ]:
vocab = np.array(encoder.get_vocabulary())
vocab[1500:]

array(['becoming', 'awkward', 'adore', 'tune', 'tones', 'suck', 'subtle',
       'stories', 'solos', 'shine', 'screaming', 'roots', 'ready',
       'pretentious', 'political', 'necessarily', 'men', 'managed',
       'kill', 'kanye', 'intriguing', 'heavily', 'halfway', 'gira',
       'father', 'ele', 'distorted', 'deliver', 'dawn', 'continue',
       'contemporary', 'comeback', 'calm', 'bom', 'became', 'acts',
       'yall', 'walk', 'tunes', 'truth', 'topics', 'spiritual', 'social',
       'sex', 'ruined', 'rain', 'painfully', 'nostalgic', 'musicians',
       'months', 'michael', 'menos', 'likely', 'intimate', 'hoping',
       'hands', 'hace', 'family', 'experiment', 'entertaining', 'endless',
       'desde', 'blends', 'bitch', 'afraid', '60', 'youll', 'wave',
       'upbeat', 'thrash', 'stars', 'places', 'obnoxious', 'mike',
       'lives', 'les', 'lazy', 'layers', 'interest', 'impact', 'forgot',
       'faves', 'evil', 'etc', 'essa', 'dude', 'drag', 'depth',
       'delicate', 'consta

In [ ]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=48,
        # Use masking to handle the variable sequence lengths
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(144)),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(1, activation='linear')
])

In [ ]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(0.005),
              metrics=['accuracy'])

In [ ]:
history = model.fit(raw_training_dataset_optimized, epochs=3,
                    validation_data=raw_validation_dataset_optimized,
                    validation_steps=30)

Epoch 1/3
310/310 ━━━━━━━━━━━━━━━━━━━━ 609s 2s/step - accuracy: 0.7146 - loss: 0.5322 - val_accuracy: 0.6708 - val_loss: 0.5566
Epoch 2/3
310/310 ━━━━━━━━━━━━━━━━━━━━ 603s 2s/step - accuracy: 0.8474 - loss: 0.3402 - val_accuracy: 0.7969 - val_loss: 0.4611
Epoch 3/3
310/310 ━━━━━━━━━━━━━━━━━━━━ 607s 2s/step - accuracy: 0.9148 - loss: 0.2242 - val_accuracy: 0.8104 - val_loss: 0.4353


In [ ]:
test_loss, test_acc = model.evaluate(raw_testing_dataset_optimized)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

97/97 ━━━━━━━━━━━━━━━━━━━━ 49s 504ms/step - accuracy: 0.8370 - loss: 0.4060
Test Loss: 0.4059697091579437
Test Accuracy: 0.836998701095581


In [ ]:
model.save('/content/drive/MyDrive/model_8370.keras')

In [ ]:
import matplotlib.pyplot as plt


def plot_graphs(history, metric):
  plt.plot(history.history[metric])
  plt.plot(history.history['val_'+metric], '')
  plt.xlabel("Epochs")
  plt.ylabel(metric)
  plt.legend([metric, 'val_'+metric])

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plot_graphs(history, 'accuracy')
plt.ylim(None, 1)
plt.subplot(1, 2, 2)
plot_graphs(history, 'loss')
plt.ylim(0, None)

In [ ]:
val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1

In [ ]:
hypermodel = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=48,
        # Use masking to handle the variable sequence lengths
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(144)),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(1, activation='linear')
])

In [ ]:
hypermodel.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                   optimizer=tf.keras.optimizers.Adam(0.005),
                   metrics=['accuracy'])

In [ ]:
hypermodel.fit(raw_training_dataset_optimized, epochs=best_epoch,
               validation_data=raw_validation_dataset_optimized,
               validation_steps=30)

In [ ]:
test_loss, test_acc = hypermodel.evaluate(raw_testing_dataset_optimized)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

In [ ]:
hypermodel.save('/content/drive/MyDrive/hypermodel.keras')